In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


In [ ]:
DATA_PATH = Path(
    os.environ.get("TELCO_DATA_PATH", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
).expanduser()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Set TELCO_DATA_PATH to the trusted CSV path; "
        "no data was generated."
    )
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.columns

In [ ]:
df.PaymentMethod.unique()

In [ ]:
df.shape

In [ ]:
df['tenure X MonthlyCharges']=df['tenure']*df['MonthlyCharges']

In [ ]:
df[['tenure','MonthlyCharges','tenure X MonthlyCharges','TotalCharges']]

In [ ]:
df.info()

In [ ]:
df.drop('customerID', axis=1, inplace=True )

In [ ]:
df.describe().T

In [ ]:
Name=[]
Data_Type=[]
Null_Values=[]
Num_of_unique=[]
Unique_Values=[]

def info_col(col):
    Name.append(col.name)
    Data_Type.append(col.dtype)
    Null_Values.append(col.isnull().sum())
    Num_of_unique.append(col.nunique())
    if col.nunique() < 10:
        Unique_Values.append(col.unique())
    else:
        Unique_Values.append('More than 10')


In [ ]:
for col in df.columns :
  info_col(df[col])
print(pd.DataFrame({'Name':Name, 'Data_Type':Data_Type,'Null_Values':Null_Values,'Num_of_unique':Num_of_unique, 'Unique_Values':Unique_Values}))

In [ ]:
df.TotalCharges.value_counts().sort_index()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df['TotalCharges'].dtype

In [ ]:
df.isna().sum()

In [ ]:
Zero_tenure=df[df['tenure']==0]
Null_TotalCharges=df[df['TotalCharges'].isna()]
print(Zero_tenure.shape)
print(Null_TotalCharges.shape)
# you will notice  [that Zero_tenure] & [Null_TotalCharge] are in same rows

In [ ]:
Null_TotalCharges

In [ ]:
df.dropna(subset=['TotalCharges'],inplace=True)

In [ ]:
df.TotalCharges.value_counts()

In [ ]:
df.duplicated().sum()

In [ ]:
cols=df.columns
cols

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
df.reset_index(inplace=True)
df.drop('index',axis=1, inplace=True)
df.head()

In [ ]:
num_col=['Partner',]

plt.figure(figsize=(20, 6))
for i,col in enumerate(num_col,1):
    plt.subplot(1,3,i)
    sns.countplot(data=df, x=df[col], hue='Dependents')
    plt.title(f'Distribution of {col}')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
num_col=['tenure','MonthlyCharges','TotalCharges']

plt.figure(figsize=(20, 6))
for i,col in enumerate(num_col,1):
    plt.subplot(1,3,i)
    sns.histplot(data=df, x=df[col], kde=True, hue='Churn')
    plt.title(f'Distribution of {col}')
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

#كل ما عدد الشهور بيزيد نسبة الل بيمشو بتقل
#اكبر فئة بتدفع فالشهر من 20 ل30 وفنفس الوقت الفئة الاكتر ولاء  ,والفئة ما بين 70 ل100 اكبر نسة بتمشي
#كل مكان مجموع الدفع على مدار الشهور اقل بتزيد نسبة انهم يمشو

In [ ]:
plt.figure(figsize=(20, 6)) #draw
plt.scatter(df['MonthlyCharges'], df['TotalCharges'])
plt.title('Monthly Charges vs Total Charges')
plt.xlabel('Monthly Charges')
plt.ylabel('Total Charges')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10)) #draw
for i,col in enumerate(num_col,1):
    plt.subplot(3,1,i)
    sns.boxplot(x=df[col],)
    plt.title(f'Distribution of {col}')
    plt.ylabel('Frequency')

plt.tight_layout(h_pad=4)
plt.show()

In [ ]:
cat_col=df.select_dtypes('object').columns
cat_col

In [ ]:
plt.figure(figsize=(24,30)) #draw
for i,col in enumerate(cat_col,1):
    plt.subplot(6,3,i)

    sns.countplot(data = df ,x = col, palette='Set2',width=0.5,)

    for p in plt.gca().patches:
        plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

    plt.title(f'Count by {col}')
plt.tight_layout(h_pad=3)
plt.show()

In [ ]:
#df.drop('tenure_group',axis=1,inplace=True)
bins = [0, 12, 24, 36, 48, 60, df['tenure'].max()]
labels = ['0-1 Year', '1-2 Years', '2-3 Years', '3-4 Years','4-5 Years', '>5 Years']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels)

In [ ]:
plt.figure(figsize=(15,5)) #draw
sns.countplot(x=df.tenure_group,data=df,palette='Set2')

for p in plt.gca().patches:
    plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

plt.title('Tenure by number of years')
plt.show()

In [ ]:
cols

In [ ]:
col_service=['MultipleLines', 'PhoneService', 'InternetService', 'OnlineSecurity', #draw
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies']

plt.figure(figsize=(20,15))
for i,col in enumerate(col_service,1):
    plt.subplot(3,3,i)
    sns.countplot(data = df ,x = col, hue = 'Churn', palette='Set1',alpha=.8)

    for p in plt.gca().patches:
        plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

    plt.title(f'Churned Customers by {col}')
plt.tight_layout(h_pad=3,w_pad=3)

In [ ]:
plt.figure(figsize=(20,5)) #draw
for i,col in enumerate(['Contract','PaymentMethod'],1):
    plt.subplot(1,2,i)
    sns.countplot(data = df ,x = col, hue = 'Churn', palette='Set1',alpha=.8)

    for p in plt.gca().patches:
        plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

    plt.title(f'Churned Customers by {col}')
plt.tight_layout(h_pad=5,w_pad=3)

In [ ]:
plt .figure(figsize=(18,5)) #draw
sns.lineplot(data = df ,x = 'tenure',y=df['tenure'].value_counts(), hue = 'Contract', palette='Set1')
plt.title('Tenure vs Contract',size=20)
plt.show()

In [ ]:
dd=df.groupby(['tenure','Contract'])['Contract'].count()
dd

In [ ]:
col_2output=['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PaperlessBilling'] #draw

plt.figure(figsize=(17,15))
for i,col in enumerate(col_2output,1):
    plt.subplot(3,3,i)
    sns.countplot(data = df ,x = col, hue = 'Churn', palette='Set1',alpha=.8)

    for p in plt.gca().patches:
        plt.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.5, int(p.get_height()), ha='center', va='center')

    plt.title(f'Churned Customers by {col}')
plt.tight_layout(h_pad=3)

In [ ]:
cat_col=df.select_dtypes('object').columns
cat_col

### **Encoding**

In [ ]:
df.replace('No internet service', 'No', inplace=True)
df.replace('No phone service', 'No', inplace=True)

In [ ]:
def replacement_yes_no(df, columns):

    for col in columns:
        df[col] = df[col].replace({'Yes': 1, 'No': 0})

columns_to_replace = [
    "Partner", "Dependents", "PhoneService",'MultipleLines', "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "PaperlessBilling", "Churn"
]

replacement_yes_no(df, columns_to_replace)

In [ ]:
df.drop(['tenure_group','tenure X MonthlyCharges','gender'],axis=1,inplace=True)

In [ ]:
categories = [['Month-to-month','One year','Two year']]
oe = OrdinalEncoder(categories = categories)
df['Contract'] = oe.fit_transform(df[['Contract']])
#pkl.dump(oe,open('Contract_oe.pkl','wb'))
df['Contract'].value_counts()

In [ ]:
encoders = {}
for col in ["InternetService", "PaymentMethod"]:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
    encoders[col] = encoder


In [ ]:
df.Churn.value_counts()

In [ ]:
df.info()

In [ ]:
corr = df.corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Apply a filter to highlight only absolute correlations less than 0.4
filtered_corr = corr.where(corr.abs() > 0.3)

# Set up the matplotlib figure
plt.figure(figsize=(24,10))

# Draw the heatmap with the mask and filtered correlation matrix
sns.heatmap(filtered_corr, mask=mask, annot=True, vmin=-1, vmax=1, cbar=True,linecolor='k',linewidths=.5, cmap='copper')

# Display the plot
plt.title('Heatmap with Upper Triangle Mask and Absolute Correlations < 0.3')
plt.show()

## **Check unbalance**

In [ ]:

plt.pie(df.Churn.value_counts(),
        labels=['Not churn','churn'],
        autopct='%1.1f%%',
        colors=['g','r'],
        shadow=True
        )
plt.title('percentage of Churn')
plt.show()

## **Splitting**

In [ ]:
x=df.drop(['Churn'], axis=1)
y=df['Churn']

## **Handle overfitting**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)
print(f"train={x_train.shape}, validation={x_val.shape}, test={x_test.shape}")


In [ ]:
# The stratified train/validation/test split is defined in the previous cell.


In [ ]:
Accuracy_models=[]
recall_models=[]
Models_names=[]
f1_score_models=[]
fpr=[]
tpr=[]

In [ ]:
def check_overfitting(model):
    y_train_pred=model.predict(x_train)
    y_val_pred=model.predict(x_val)
    y_test_pred=model.predict(x_test)

    model_name = model.__class__.__name__ # store model name
    print(f'You Applyed [{model_name}] Model')
    print('='*30)

    Train_accuracy = accuracy_score(y_train,y_train_pred)
    Valid_accuracy = accuracy_score(y_val,y_val_pred)
    Test_accuracy = accuracy_score(y_test,y_test_pred)

    print(f"-Train accuracy: {Train_accuracy*100:.2f}%")
    print(f"-Validation accuracy: {Valid_accuracy*100:.2f}%")
    print(f"-Test accuracy: {Test_accuracy*100:.2f}%")
#Evaluation
    f1_score_test=f1_score(y_test,y_test_pred)
    precision_score_test=precision_score(y_test,y_test_pred)
    recall_score_test=recall_score(y_test,y_test_pred)

    print(f'-F1_Score: {f1_score_test*100:.2f}%')
    print(f'-Precision: {precision_score_test*100:.2f}%')
    print(f'-Recall: {recall_score_test*100:.2f}%\n\n')

    plt.figure(figsize=(10,5))
# Confusion Matrix
    plt.subplot(1, 2, 1)
    sns.heatmap(confusion_matrix(y_test, y_test_pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')

# ROC Curve

    plt.subplot(1, 2, 2)

    y_test_prob = model.predict_proba(x_test)[:, 1]
    fp, tp, _ = roc_curve(y_test, y_test_prob)
#
    plt.plot([0, 1], [0, 1], 'k--')
    plt.plot(fp, tp, color="r")
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')

    plt.tight_layout()
    plt.show()

    y_test_prob = model.predict_proba(x_test)[:, 1]#
    fpr1, tpr1, _ = roc_curve(y_test,y_test_prob)#


    if abs(Valid_accuracy - Train_accuracy)*100 > 5:
        print('='*20)
        print(f'Unfotionatly you have Overfitting more than with Error=: {abs(Valid_accuracy - Train_accuracy)*100:.2f}%\n')
        print('='*20)
    else:
        print('='*20)
        print(f"-Not Overfitting with Test Accuracy : {Test_accuracy*100:.2f}%\n\n")
        print('='*20)

    recall_models.append(recall_score_test)
    Accuracy_models.append(Test_accuracy)
    f1_score_models.append(f1_score_test)
    Models_names.append(model_name)
    fpr.append(fpr1)#
    tpr.append(tpr1)#

## **[1]-Decision tree**

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=42)

params={'criterion':['gini', 'entropy','log_loss'],
        'max_depth':[5, 6, 7, 8 ,9, 10, 11, 12],
        'ccp_alpha':[0.0001,0.001, 0.01, 0.02, 0.03]
        }

grid=GridSearchCV(estimator=decision_tree,
                  param_grid=params,
                  cv=5,
                  scoring='accuracy')

grid.fit(x_train,y_train)

In [ ]:
finall_decision_tree=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(finall_decision_tree)

## **Scaling**

In [ ]:
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_val=sc.transform(x_val)
x_test=sc.transform(x_test)

## **[2]-LogisticRegression**

In [ ]:
lr= LogisticRegression()
params={'multi_class':['ovr','auto'],
        'C':[0.1,0.2,0.3,1,5,10],
        'penalty':['l1','l2','l3']
        }

grid=GridSearchCV(estimator=lr,
                  param_grid=params,
                  cv=5,
                  scoring='accuracy')
grid.fit(x_train,y_train)

In [ ]:
finall_lr=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(finall_lr)

## **[3]-SVC**

In [ ]:
svc=SVC(probability=True,random_state=42)

svc.fit(x_train,y_train)

In [ ]:
check_overfitting(svc)

### **[4]-Random_forest**

In [ ]:
random_forest=RandomForestClassifier(random_state=42,ccp_alpha=0.001)

random_forest.fit(x_train,y_train)

In [ ]:
check_overfitting(random_forest)

# [5]-Gradient_boosting

In [ ]:
gradient_boosting=GradientBoostingClassifier(n_estimators=200,
                                            learning_rate=0.001,
                                            max_depth=4,
                                            random_state=42)
gradient_boosting.fit(x_train,y_train)

In [ ]:
check_overfitting(gradient_boosting)

# [6]-xgboost

In [ ]:
xgboost = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_estimators=115,
    max_depth=4,
    learning_rate=0.2,
    subsample=1.0,
    colsample_bytree=1.0,
)
xgboost.fit(x_train, y_train)


In [ ]:
check_overfitting(xgboost)

# [7]-AdaBoost

In [ ]:
adaboost = AdaBoostClassifier(
    random_state=42,
    n_estimators=200,
    learning_rate=1.0,
)
adaboost.fit(x_train, y_train)


In [ ]:
check_overfitting(adaboost)

#[8]-Stacking

In [ ]:
base_models = [
    ("rf", RandomForestClassifier(max_depth=4, n_estimators=40, random_state=42)),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.01, max_depth=4, random_state=42
        ),
    ),
]
meta_model = XGBClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.01, random_state=42, eval_metric="logloss"
)
stacking = StackingClassifier(estimators=base_models, final_estimator=meta_model)
stacking.fit(x_train, y_train)


In [ ]:
check_overfitting(stacking)

# [9]-Gaussian NB

In [ ]:
GNB=GaussianNB()
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}
grid=GridSearchCV(estimator=GNB,
                  param_grid=param_grid,
                  cv=5,
                  scoring='recall')
grid.fit(x_train,y_train)

In [ ]:
Nave_Modle=grid.best_estimator_
grid.best_params_

In [ ]:
check_overfitting(Nave_Modle)

# [10]-KNN

In [ ]:
knn=KNeighborsClassifier(algorithm='auto',weights='uniform',p=1,n_neighbors=13)

knn.fit(x_train,y_train)

In [ ]:
check_overfitting(knn)

In [ ]:
print(f1_score_models)

# **Accuracy Comparison**

In [ ]:
model_accuracy=pd.DataFrame({'Model':Models_names[:10],'Accuracy':Accuracy_models[:10],'Recall':recall_models[:10],'F1_Score':f1_score_models[:10],'fpr':fpr[:10],'tpr':tpr[:10]}).sort_values(by='Accuracy',ascending=False)
model_accuracy

In [ ]:
colors = ['red','green','blue','orange','purple','brown','black','magenta','olive','cyan','yellow']

plt.figure(figsize=(10,8))



for fp, tp, color, name in zip(fpr, tpr, colors, Models_names):
    plt.plot(fp, tp ,color=color, label=name)

plt.plot([0, 1], [0, 1],'k--')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve for Multiple Models', fontsize=14)


plt.legend(loc='lower right')


plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Accuracy']*100,marker='o',color='b',lw=2,label='Accuracy')
for i, score in enumerate(model_accuracy['Accuracy']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='b')

sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['Recall']*100,marker='o',color='r',lw=2,label='Recall')
for i, score in enumerate(model_accuracy['Recall']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='r')

sns.lineplot(x=model_accuracy['Model'],y=model_accuracy['F1_Score']*100,marker='o',color='g',lw=2,label='F1-Score')
for i, score in enumerate(model_accuracy['F1_Score']):
    plt.text(i, score*100 + 0.2, f'{score*100:.1f}', ha='center', color='g')
#---------------------------------------------------------------

plt.legend()
plt.xticks(rotation=45)

plt.xlabel('Models Names')
plt.ylabel('Accuracy')
plt.title('Accuracy of models')
plt.show()

In [ ]:
best_accuracy=model_accuracy.sort_values(by='Accuracy',ascending=False).reset_index(drop=True)
best_recall=model_accuracy.sort_values(by='Recall',ascending=False).reset_index(drop=True)
best_f1_score=model_accuracy.sort_values(by='F1_Score',ascending=False).reset_index(drop=True)

In [ ]:
print(f'Best Accuracy Model: {best_accuracy.Model[0]} With accuracy: {best_accuracy.Accuracy[0]*100:0.2f}%')
print(f'Best Recall Model: {best_recall.Model[0]} With Recall: {best_recall.Recall[0]*100:0.2f}%')
print(f'Best F1_Score Model: {best_f1_score.Model[0]} With F1_Score: {best_f1_score.F1_Score[0]*100:0.2f}%')